In [ ]:
import os
import time
import random
import urllib.request

import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [ ]:
URL_MODELO = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"

if not os.path.exists("face_landmarker.task"):
    urllib.request.urlretrieve(URL_MODELO, "face_landmarker.task")
print("Modelo face_landmarker.task disponible.")

In [ ]:
ANCHO, ALTO = 640, 480
RUTA_MODELO = "face_landmarker.task"
SALIDA_VIDEO = "espejo_feria.mp4"
FPS_GRABACION = 25
DURACION_MAXIMA = 30.0

OBJETOS = ["GAFAS_THUG", "CORONA", "PAYASO", "ALIEN"]
UMBRAL_SORPRESA, UMBRAL_SONRISA, UMBRAL_CENO = 0.40, 0.42, 0.38
UMBRAL_PARPADEO, UMBRAL_MANTENER_PARPADEO = 0.40, 0.30
SEGUNDOS_PARPADEO, SEGUNDOS_OBJETO, SEGUNDOS_CALIBRACION = 1.5, 5.0, 2.0
SUAVIZADO_GEOMETRIA, SUAVIZADO_INTENSIDAD = 0.45, 0.28
FRAMES_SIN_CARA = 8

NARIZ = 1
OJO_IZQ = (33, 133)
OJO_DER = (362, 263)
EXPRESIONES = ("eyeBlinkLeft", "eyeBlinkRight", "mouthSmileLeft", "mouthSmileRight", "browDownLeft",
               "browDownRight", "browInnerUp", "browOuterUpLeft", "browOuterUpRight", "jawOpen")

BASE_X, BASE_Y = np.meshgrid(np.arange(ANCHO, dtype=np.float32), np.arange(ALTO, dtype=np.float32))
CACHE_SPRITES = {}


def lente_radial(map_x, map_y, centro, radio, potencia):
    dx, dy = map_x - centro[0], map_y - centro[1]
    dist = np.hypot(dx, dy)
    mascara = dist < radio
    nd = dist[mascara] / radio
    escala = (nd ** (1.0 + potencia)) / (nd + 1e-6)
    map_x[mascara] = centro[0] + dx[mascara] * escala
    map_y[mascara] = centro[1] + dy[mascara] * escala


def deformar(frame, map_x, map_y):
    return cv2.remap(frame, map_x, map_y, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)


def filtro_sonrisa(frame, nariz, ojos, dist_ojos, intensidad):
    map_x, map_y = BASE_X.copy(), BASE_Y.copy()
    lente_radial(map_x, map_y, nariz, dist_ojos * 2.15, intensidad * 0.95)
    frame = deformar(frame, map_x, map_y)
    capa = np.full_like(frame, (30, 140, 255))
    for lado in (-1, 1):
        mejilla = (int(nariz[0] + lado * dist_ojos * 0.65), int(nariz[1] + dist_ojos * 0.25))
        cv2.circle(capa, mejilla, max(10, int(dist_ojos * 0.28)), (130, 110, 255), -1)
    peso = 0.16 * intensidad
    return cv2.addWeighted(capa, peso, frame, 1.0 - peso, 0)


def filtro_ceno(frame, nariz, ojos, dist_ojos, intensidad):
    map_x, map_y = BASE_X.copy(), BASE_Y.copy()
    lente_radial(map_x, map_y, nariz, dist_ojos * 2.15, -intensidad * 0.40)
    frame = deformar(frame, map_x, map_y)
    gris = cv2.cvtColor(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), cv2.COLOR_GRAY2BGR)
    frame = cv2.addWeighted(gris, 0.45 * intensidad, frame, 1.0 - 0.45 * intensidad, 0)
    distancia = np.hypot(BASE_X - nariz[0], BASE_Y - nariz[1])
    vineta = np.clip(1.15 - (distancia / (ANCHO * 0.65)) * 0.65 * intensidad, 0.35, 1.0)
    return (frame * vineta[..., None]).astype(np.uint8)


def filtro_sorpresa(frame, nariz, ojos, dist_ojos, intensidad):
    map_x, map_y = BASE_X.copy(), BASE_Y.copy()
    for ojo in ojos:
        lente_radial(map_x, map_y, ojo, dist_ojos * 0.52, intensidad * 1.15)
    frame = deformar(frame, map_x, map_y)
    if intensidad > 0.45:
        for angulo in np.radians(range(0, 360, 15)):
            inicio = (int(ANCHO / 2 + np.cos(angulo) * ANCHO * 0.42), int(ALTO / 2 + np.sin(angulo) * ALTO * 0.42))
            fin = (int(ANCHO / 2 + np.cos(angulo) * ANCHO * 0.75), int(ALTO / 2 + np.sin(angulo) * ALTO * 0.75))
            cv2.line(frame, inicio, fin, (255, 255, 255), 2)
    return frame


FILTROS = {"SONRISA": filtro_sonrisa, "CENO": filtro_ceno, "SORPRESA": filtro_sorpresa}


def aplicar_filtro(frame, nariz, ojos, dist_ojos, modo, intensidad):
    if modo not in FILTROS or intensidad <= 0.02:
        return frame
    return FILTROS[modo](frame, nariz, ojos, dist_ojos, intensidad)


def referencia(c, u):
    return lambda dx, dy: (int(c + dx * u), int(c + dy * u))


def sprite_gafas(obj, c, u):
    mitad = [
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 1, 2, 3, 2, 2, 3, 2, 2, 2, 2, 1, 0, 0],
        [0, 0, 1, 2, 2, 3, 2, 2, 3, 2, 2, 2, 1, 0, 0],
        [0, 0, 1, 2, 2, 2, 4, 2, 2, 4, 2, 2, 1, 0, 0],
        [0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
    ]
    paleta = {1: (15, 15, 20, 255), 2: (40, 42, 52, 235), 3: (255, 255, 255, 255), 4: (180, 190, 205, 245)}
    lado = max(2, int(u * 0.086))
    y0 = int(c - 2.3 * lado)
    for fila, valores in enumerate(mitad):
        for columna, valor in enumerate(valores):
            if valor == 0:
                continue
            for signo in (-1, 1):
                if columna == 0 and signo == -1:
                    continue
                x = c + signo * columna * lado - lado // 2
                y = y0 + fila * lado
                cv2.rectangle(obj, (x, y), (x + lado, y + lado), paleta[valor], -1)


def sprite_corona(obj, c, u):
    pt = referencia(c, u)
    grosor = max(2, int(u * 0.06))
    terciopelo = np.array([pt(-0.78, -1.10), pt(-0.65, -1.65), pt(0, -1.85), pt(0.65, -1.65), pt(0.78, -1.10)], np.int32)
    cv2.fillPoly(obj, [terciopelo], (30, 15, 130, 255), cv2.LINE_AA)

    corona = np.array([
        pt(-0.88, -1.02), pt(-1.02, -1.72), pt(-0.70, -1.36), pt(-0.48, -1.88),
        pt(-0.22, -1.40), pt(0.00, -2.04), pt(0.22, -1.40), pt(0.48, -1.88),
        pt(0.70, -1.36), pt(1.02, -1.72), pt(0.88, -1.02),
    ], np.int32)
    cv2.fillPoly(obj, [corona], (15, 180, 245, 255), cv2.LINE_AA)

    brillo = np.array([pt(-0.34, -1.02), pt(-0.22, -1.40), pt(0.00, -2.04), pt(0.22, -1.40), pt(0.34, -1.02)], np.int32)
    cv2.fillPoly(obj, [brillo], (65, 225, 255, 255), cv2.LINE_AA)
    cv2.polylines(obj, [corona], True, (10, 115, 175, 255), grosor, cv2.LINE_AA)

    cv2.rectangle(obj, pt(-0.88, -1.16), pt(0.88, -0.98), (15, 155, 225, 255), -1)
    cv2.rectangle(obj, pt(-0.88, -1.16), pt(0.88, -0.98), (10, 110, 170, 255), max(1, grosor - 1))
    for x in np.linspace(-0.72, 0.72, 7):
        cv2.circle(obj, pt(x, -1.07), max(2, int(u * 0.04)), (245, 250, 255, 255), -1, cv2.LINE_AA)

    for punta in [(-1.02, -1.72), (-0.48, -1.88), (0.00, -2.04), (0.48, -1.88), (1.02, -1.72)]:
        radio = max(3, int(u * 0.085))
        cv2.circle(obj, pt(*punta), radio, (30, 210, 255, 255), -1, cv2.LINE_AA)
        cv2.circle(obj, pt(*punta), radio, (10, 115, 175, 255), 1, cv2.LINE_AA)

    rubi = np.array([pt(0, -1.55), pt(0.13, -1.40), pt(0, -1.25), pt(-0.13, -1.40)], np.int32)
    cv2.fillPoly(obj, [rubi], (45, 30, 235, 255), cv2.LINE_AA)
    cv2.polylines(obj, [rubi], True, (20, 10, 140, 255), 2, cv2.LINE_AA)
    cv2.circle(obj, pt(-0.03, -1.44), max(2, int(u * 0.03)), (255, 255, 255, 255), -1)

    for x, color in [(-0.54, (230, 150, 20)), (0.54, (50, 200, 60))]:
        radio = max(4, int(u * 0.09))
        cv2.circle(obj, pt(x, -1.35), radio, (*color, 255), -1, cv2.LINE_AA)
        cv2.circle(obj, pt(x, -1.35), radio, (20, 60, 80, 255), 1, cv2.LINE_AA)
        cv2.circle(obj, pt(x - 0.03, -1.38), max(1, radio // 3), (255, 255, 255, 240), -1)


def sprite_payaso(obj, c, u):
    pt = referencia(c, u)
    radio = max(9, int(u * 0.33))
    centro = pt(0, 0.50)
    cv2.circle(obj, centro, radio, (15, 15, 165, 255), -1, cv2.LINE_AA)
    cv2.circle(obj, (centro[0] - int(radio * 0.12), centro[1] - int(radio * 0.12)), int(radio * 0.84), (30, 30, 230, 255), -1, cv2.LINE_AA)
    cv2.circle(obj, (centro[0] - int(radio * 0.22), centro[1] - int(radio * 0.22)), int(radio * 0.52), (65, 95, 255, 255), -1, cv2.LINE_AA)
    cv2.ellipse(obj, (centro[0] - int(radio * 0.34), centro[1] - int(radio * 0.34)),
                (max(2, int(radio * 0.26)), max(1, int(radio * 0.14))), -35, 0, 360, (255, 255, 255, 240), -1, cv2.LINE_AA)

    gorro = np.zeros_like(obj)
    cono = np.array([pt(-0.68, -1.02), pt(0.68, -1.02), pt(0, -2.26)], np.int32)
    cv2.fillPoly(gorro, [cono], (225, 75, 155, 255), cv2.LINE_AA)
    cv2.ellipse(gorro, pt(0, -1.02), (max(10, int(0.68 * u)), max(4, int(0.15 * u))), 0, 0, 180, (225, 75, 155, 255), -1, cv2.LINE_AA)

    silueta = gorro[:, :, 3] > 0
    lunares = gorro.copy()
    for x, y, color in [(-0.22, -1.25, (0, 230, 255)), (0.24, -1.22, (100, 255, 120)), (0.0, -1.52, (255, 230, 80)),
                        (-0.14, -1.82, (80, 180, 255)), (0.14, -1.78, (0, 230, 255))]:
        cv2.circle(lunares, pt(x, y), max(3, int(u * 0.085)), (*color, 255), -1, cv2.LINE_AA)
    gorro[silueta] = lunares[silueta]
    np.maximum(obj, gorro, out=obj)

    for x in np.linspace(-0.64, 0.64, 9):
        y = -1.02 + 0.12 * np.cos(x * np.pi / 1.36)
        cv2.circle(obj, pt(x, y), max(4, int(u * 0.11)), (245, 250, 255, 255), -1, cv2.LINE_AA)
        cv2.circle(obj, pt(x, y), max(4, int(u * 0.11)), (190, 200, 215, 255), 1, cv2.LINE_AA)

    pompon = max(6, int(u * 0.16))
    for angulo in np.radians(range(0, 360, 60)):
        cv2.circle(obj, (int(c + np.cos(angulo) * pompon * 0.65), int(c - 2.30 * u + np.sin(angulo) * pompon * 0.65)),
                   int(pompon * 0.72), (0, 215, 255, 255), -1, cv2.LINE_AA)
    cv2.circle(obj, pt(0, -2.30), int(pompon * 0.8), (80, 245, 255, 255), -1, cv2.LINE_AA)


def sprite_alien(obj, c, u):
    pt = referencia(c, u)
    t = np.linspace(0, 1, 18)[:, None]
    for lado in (-1, 1):
        cv2.ellipse(obj, pt(lado * 0.42, -1.00), (max(5, int(0.16 * u)), max(3, int(0.08 * u))),
                    lado * 12, 0, 360, (30, 150, 40, 255), -1, cv2.LINE_AA)

        p0, p1, p2 = np.array([lado * 0.42, -1.00]), np.array([lado * 0.96, -1.48]), np.array([lado * 0.76, -1.98])
        curva = (1 - t) ** 2 * p0 + 2 * (1 - t) * t * p1 + t ** 2 * p2
        puntos = np.array([pt(x, y) for x, y in curva], np.int32)
        cv2.polylines(obj, [puntos], False, (20, 115, 30, 255), max(4, int(u * 0.11)), cv2.LINE_AA)
        cv2.polylines(obj, [puntos], False, (75, 235, 95, 255), max(2, int(u * 0.06)), cv2.LINE_AA)

        punta = pt(lado * 0.76, -1.98)
        radio = max(7, int(u * 0.21))
        cv2.circle(obj, punta, int(radio * 1.55), (90, 255, 130, 75), -1, cv2.LINE_AA)
        cv2.circle(obj, punta, radio, (55, 235, 95, 255), -1, cv2.LINE_AA)
        cv2.circle(obj, punta, int(radio * 0.65), (165, 255, 190, 255), -1, cv2.LINE_AA)
        cv2.circle(obj, (punta[0] - int(radio * 0.28), punta[1] - int(radio * 0.28)), max(2, int(radio * 0.24)), (255, 255, 255, 245), -1, cv2.LINE_AA)


CONSTRUCTORES = {"GAFAS_THUG": sprite_gafas, "CORONA": sprite_corona, "PAYASO": sprite_payaso, "ALIEN": sprite_alien}


def anadir_sombra(obj, u):
    desplazamiento = max(2, int(u * 0.06))
    alfa_sombra = cv2.GaussianBlur(obj[:, :, 3], (0, 0), max(1.5, u * 0.05)) * 0.42
    lienzo = np.zeros_like(obj)
    lienzo[desplazamiento:, :, 3] = alfa_sombra[:-desplazamiento].astype(np.uint8)
    alfa_obj = obj[:, :, 3:4] / 255.0
    alfa_som = lienzo[:, :, 3:4] / 255.0
    lienzo[:, :, :3] = (obj[:, :, :3] * alfa_obj).astype(np.uint8)
    lienzo[:, :, 3:4] = ((alfa_obj + alfa_som * (1.0 - alfa_obj)) * 255.0).astype(np.uint8)
    return lienzo


def obtener_sprite(tipo, u):
    u = max(8, int(round(u / 2.0)) * 2)
    clave = (tipo, u)
    if clave not in CACHE_SPRITES:
        if len(CACHE_SPRITES) > 96:
            CACHE_SPRITES.clear()
        tam = int(u * 5.6)
        obj = np.zeros((tam, tam, 4), dtype=np.uint8)
        CONSTRUCTORES[tipo](obj, tam // 2, u)
        CACHE_SPRITES[clave] = anadir_sombra(obj, u)
    return CACHE_SPRITES[clave]


def pegar_objeto(frame, centro, u, angulo, tipo):
    if not tipo or u * 5.6 < 20:
        return frame
    sprite = obtener_sprite(tipo, u)
    tam = sprite.shape[0]
    c = tam // 2
    rotado = cv2.warpAffine(sprite, cv2.getRotationMatrix2D((c, c), angulo, 1.0), (tam, tam))
    x1, y1 = centro[0] - c, centro[1] - c
    fx1, fy1, fx2, fy2 = max(0, x1), max(0, y1), min(ANCHO, x1 + tam), min(ALTO, y1 + tam)
    if fx1 < fx2 and fy1 < fy2:
        recorte = rotado[fy1 - y1:fy2 - y1, fx1 - x1:fx2 - x1]
        alfa = recorte[:, :, 3:4] / 255.0
        fondo = frame[fy1:fy2, fx1:fx2]
        frame[fy1:fy2, fx1:fx2] = np.clip(recorte[:, :, :3] + (1.0 - alfa) * fondo, 0, 255).astype(np.uint8)
    return frame


def suavizar(anterior, nuevo, peso):
    if anterior is None:
        return nuevo
    return tuple(peso * n + (1.0 - peso) * a for a, n in zip(anterior, nuevo))


def texto(frame, contenido, posicion, color=(255, 255, 255), escala=0.6):
    grosor = 2 if escala >= 0.6 else 1
    cv2.putText(frame, contenido, posicion, cv2.FONT_HERSHEY_SIMPLEX, escala, (0, 0, 0), grosor + 3, cv2.LINE_AA)
    cv2.putText(frame, contenido, posicion, cv2.FONT_HERSHEY_SIMPLEX, escala, color, grosor, cv2.LINE_AA)


class EspejoInteractivo:
    def __init__(self):
        opciones = vision.FaceLandmarkerOptions(
            base_options=python.BaseOptions(model_asset_path=RUTA_MODELO),
            running_mode=vision.RunningMode.VIDEO,
            output_face_blendshapes=True,
            num_faces=3,
        )
        self.detector = vision.FaceLandmarker.create_from_options(opciones)
        self.inicio = time.perf_counter()
        self.ultimo_ts = -1
        self.mostrar_malla = False
        self.mostrar_panel = True
        self.modo, self.intensidad = "NORMAL", 0.0
        self.objeto, self.fin_objeto = None, 0.0
        self.cierre, self.sorteado = None, False
        self.geometria = None
        self.frames_sin_cara = 0
        self.fps = 0.0
        self.ultimo_instante = None
        self.valores = {}
        self.recalibrar()

    def recalibrar(self):
        self.base = None
        self.muestras = {clave: [] for clave in EXPRESIONES}
        self.inicio_calibracion = None

    @property
    def calibrando(self):
        return self.base is None

    def detectar(self, frame):
        imagen = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        marca = max(self.ultimo_ts + 1, int((time.perf_counter() - self.inicio) * 1000))
        self.ultimo_ts = marca
        return self.detector.detect_for_video(imagen, marca)

    @staticmethod
    def puntos_clave(landmarks):
        medio = lambda i, j: ((landmarks[i].x + landmarks[j].x) * ANCHO / 2, (landmarks[i].y + landmarks[j].y) * ALTO / 2)
        izquierdo, derecho = medio(*OJO_IZQ), medio(*OJO_DER)
        nariz = (landmarks[NARIZ].x * ANCHO, landmarks[NARIZ].y * ALTO)
        distancia = float(np.hypot(derecho[0] - izquierdo[0], derecho[1] - izquierdo[1]))
        angulo = -float(np.degrees(np.arctan2(derecho[1] - izquierdo[1], derecho[0] - izquierdo[0])))
        return nariz, izquierdo, derecho, distancia, angulo

    def elegir_cara(self, resultado):
        mejor, mayor = None, 0.0
        for indice, landmarks in enumerate(resultado.face_landmarks):
            distancia = self.puntos_clave(landmarks)[3]
            if distancia > mayor and indice < len(resultado.face_blendshapes):
                mejor, mayor = indice, distancia
        return mejor

    def leer_expresiones(self, categorias, ahora):
        crudo = {c.category_name: c.score for c in categorias}
        if self.calibrando:
            if self.inicio_calibracion is None:
                self.inicio_calibracion = ahora
            for clave in EXPRESIONES:
                self.muestras[clave].append(crudo.get(clave, 0.0))
            if ahora - self.inicio_calibracion >= SEGUNDOS_CALIBRACION:
                self.base = {clave: float(np.mean(valores)) for clave, valores in self.muestras.items()}
            return None
        return {clave: max(0.0, crudo.get(clave, 0.0) - self.base[clave]) for clave in EXPRESIONES}

    @staticmethod
    def clasificar(v):
        sonrisa = (v["mouthSmileLeft"] + v["mouthSmileRight"]) / 2
        ceno = (v["browDownLeft"] + v["browDownRight"]) / 2
        cejas = max(v["browInnerUp"], (v["browOuterUpLeft"] + v["browOuterUpRight"]) / 2)
        sorpresa = min(cejas, v["jawOpen"] * 1.2)
        if sorpresa > UMBRAL_SORPRESA:
            return "SORPRESA", min(1.0, (sorpresa - 0.25) * 1.8)
        if sonrisa > UMBRAL_SONRISA:
            return "SONRISA", min(1.0, (sonrisa - 0.25) * 1.35)
        if ceno > UMBRAL_CENO:
            return "CENO", min(1.0, (ceno - 0.20) * 1.5)
        return "NORMAL", 0.0

    def actualizar_expresion(self, v):
        modo, objetivo = self.clasificar(v)
        if modo != "NORMAL" and modo != self.modo and self.intensidad > 0.15:
            objetivo = 0.0
        elif modo != "NORMAL":
            self.modo = modo
        self.intensidad = (1.0 - SUAVIZADO_INTENSIDAD) * self.intensidad + SUAVIZADO_INTENSIDAD * objetivo

    def actualizar_objeto(self, v, ahora):
        umbral = UMBRAL_MANTENER_PARPADEO if self.cierre is not None else UMBRAL_PARPADEO
        if v["eyeBlinkLeft"] > umbral and v["eyeBlinkRight"] > umbral:
            self.cierre = self.cierre if self.cierre is not None else ahora
            if ahora - self.cierre >= SEGUNDOS_PARPADEO and not self.sorteado:
                self.objeto = random.choice([o for o in OBJETOS if o != self.objeto])
                self.fin_objeto, self.sorteado = ahora + SEGUNDOS_OBJETO, True
        else:
            self.cierre, self.sorteado = None, False
        if ahora > self.fin_objeto:
            self.objeto = None

    def perder_cara(self):
        self.frames_sin_cara += 1
        self.intensidad *= 0.8
        if self.frames_sin_cara >= FRAMES_SIN_CARA:
            self.geometria = None
            self.objeto = None
            self.cierre, self.sorteado = None, False

    def procesar(self, frame, ahora):
        resultado = self.detectar(frame)
        indice = self.elegir_cara(resultado)
        if indice is None:
            self.perder_cara()
            if self.geometria is not None:
                frame = self.escena(frame)
            return self.dibujar_panel(frame, "Sin cara detectada")

        self.frames_sin_cara = 0
        landmarks = resultado.face_landmarks[indice]
        nariz, izquierdo, derecho, distancia, angulo = self.puntos_clave(landmarks)
        centro = ((izquierdo[0] + derecho[0]) / 2, (izquierdo[1] + derecho[1]) / 2)
        self.geometria = suavizar(self.geometria, (*nariz, *izquierdo, *derecho, *centro, distancia, angulo), SUAVIZADO_GEOMETRIA)

        v = self.leer_expresiones(resultado.face_blendshapes[indice], ahora)
        if v is None:
            falta = max(0.0, SEGUNDOS_CALIBRACION - (ahora - self.inicio_calibracion))
            return self.dibujar_panel(frame, f"Calibrando, cara neutra: {falta:0.1f} s")

        self.valores = v
        self.actualizar_expresion(v)
        self.actualizar_objeto(v, ahora)
        return self.dibujar_panel(self.escena(frame, landmarks), None)

    def escena(self, frame, landmarks=None):
        g = self.geometria
        nariz = (int(g[0]), int(g[1]))
        ojos = [(int(g[2]), int(g[3])), (int(g[4]), int(g[5]))]
        centro = (int(g[6]), int(g[7]))
        frame = pegar_objeto(frame, centro, g[8], g[9], self.objeto)
        if self.mostrar_malla and landmarks is not None:
            for p in landmarks[::3]:
                cv2.circle(frame, (int(p.x * ANCHO), int(p.y * ALTO)), 1, (0, 255, 180), -1)
        return aplicar_filtro(frame, nariz, ojos, g[8], self.modo, self.intensidad)

    def actualizar_fps(self):
        instante = time.perf_counter()
        if self.ultimo_instante is not None:
            actual = 1.0 / max(instante - self.ultimo_instante, 1e-6)
            self.fps = actual if self.fps == 0.0 else 0.9 * self.fps + 0.1 * actual
        self.ultimo_instante = instante

    def dibujar_panel(self, frame, mensaje):
        if not self.mostrar_panel:
            return frame
        if mensaje:
            texto(frame, mensaje, (12, 28), (0, 220, 255), 0.7)
        else:
            activo = self.modo if self.intensidad > 0.05 else "NORMAL"
            texto(frame, f"Expresion: {activo}  {self.intensidad:0.2f}", (12, 28), (0, 255, 180), 0.7)
            if self.objeto:
                texto(frame, f"Objeto: {self.objeto}", (12, 56), (255, 220, 0), 0.6)
            elif self.cierre is not None:
                texto(frame, f"Ojos cerrados: {time.time() - self.cierre:0.1f} / {SEGUNDOS_PARPADEO} s", (12, 56), (255, 220, 0), 0.6)
        texto(frame, f"{self.fps:0.0f} FPS", (ANCHO - 90, 28))
        texto(frame, "ESC salir | L malla | C calibrar | H panel | R grabar", (12, ALTO - 12), (220, 220, 220), 0.5)
        return frame


def abrir_camara(indice):
    cap = cv2.VideoCapture(indice)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, ANCHO)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, ALTO)
    return cap


def ajustar_formato(frame):
    alto, ancho = frame.shape[:2]
    proporcion = ANCHO / ALTO
    if ancho / alto > proporcion:
        nuevo = int(alto * proporcion)
        x0 = (ancho - nuevo) // 2
        frame = frame[:, x0:x0 + nuevo]
    elif ancho / alto < proporcion:
        nuevo = int(ancho / proporcion)
        y0 = (alto - nuevo) // 2
        frame = frame[y0:y0 + nuevo]
    if frame.shape[1] != ANCHO or frame.shape[0] != ALTO:
        frame = cv2.resize(frame, (ANCHO, ALTO))
    return frame


def ejecutar(indice_camara=0):
    cap = abrir_camara(indice_camara)
    if not cap.isOpened():
        raise RuntimeError("No se pudo abrir la camara")
    espejo = EspejoInteractivo()
    grabador, inicio_grabacion, escritos = None, 0.0, 0

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            ahora = time.time()
            espejo.actualizar_fps()
            frame = cv2.flip(ajustar_formato(frame), 1)
            frame = espejo.procesar(frame, ahora)

            if grabador is not None:
                if ahora - inicio_grabacion >= DURACION_MAXIMA:
                    grabador.release()
                    grabador = None
                else:
                    pendientes = int((ahora - inicio_grabacion) * FPS_GRABACION) + 1 - escritos
                    for _ in range(max(0, pendientes)):
                        grabador.write(frame)
                        escritos += 1
                    cv2.circle(frame, (ANCHO - 20, 50), 7, (0, 0, 255), -1)

            cv2.imshow("Espejo interactivo", frame)
            tecla = cv2.waitKey(1) & 0xFF
            if tecla == 27:
                break
            if tecla in (ord("l"), ord("L")):
                espejo.mostrar_malla = not espejo.mostrar_malla
            if tecla in (ord("h"), ord("H")):
                espejo.mostrar_panel = not espejo.mostrar_panel
            if tecla in (ord("c"), ord("C")):
                espejo.recalibrar()
            if tecla in (ord("r"), ord("R")):
                if grabador is None:
                    grabador = cv2.VideoWriter(SALIDA_VIDEO, cv2.VideoWriter_fourcc(*"mp4v"), FPS_GRABACION, (ANCHO, ALTO))
                    inicio_grabacion, escritos = ahora, 0
                else:
                    grabador.release()
                    grabador = None
    finally:
        if grabador is not None:
            grabador.release()
        cap.release()
        cv2.destroyAllWindows()
        espejo.detector.close()


ejecutar()